In [3]:
# 系统提示词
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.tools import tool

# 1.加载环境变量
load_dotenv()

# 2.初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# =================方式二：基于文档注释描述工具==================
# 3.定义tool
@tool
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    获取指定位置当前时间的天气以及未来的天气(可选)
    :param location: 城市名称
    :param units: 温度单位(摄氏度或华氏度)['celsius', 'fahrenheit']
    :param include_forecast: 是否包含未来天气预报
    :return: 天气信息
    """
    temp = 22 if units == "celsius" else 72
    result = f"当前{location}的温度为:{temp} ° {units[0].upper()}"
    if include_forecast:
        result += "\n未来5天的天气: 晴天"
    return result


tools = [get_weather]

# 4.创建智能体
agent = create_agent(
    model=model,
    tools=tools,
)

# 5.流式调用
stream = agent.stream_events(
    {"messages": [HumanMessage(content="北京未来5天天气怎么样？用华氏温度")]},
    version="v3"
)

for message in stream.messages:
    for text in message.text:
        print(text, end="", flush=True)

我来帮您查询北京未来5天的天气情况，使用华氏温度。我来为您查询了北京的天气情况：

## ☀️ 北京未来5天天气

**当前温度：72°F**

**未来5天天气：晴天** ☀️

从天气信息来看，北京未来5天都将是**晴好天气**，非常适合出行和户外活动！

### 温馨提示：
- 晴天气温相对舒适（当前为72°F，约22°C），体感比较宜人
- 建议您出门时注意防晒，随身携带饮用水
- 早晚温差可能较大，可以随身带一件薄外套

如果您需要了解更多细节（比如每天的具体最高/最低温度、风力等），欢迎告诉我，我可以进一步为您查询！